<a href="https://colab.research.google.com/github/Rajeeb321123/Pytorch/blob/master/08_0_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning embedding layer in Neural network

## Term Frequency-Inverse Document Frequency (TF-IDF)

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

where:

$$
\text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}
$$

$$
\text{IDF}(t) = \log \left( \frac{1 + N}{1 + \text{DF}(t)} \right)
$$

- $t$: Term (word)
- $d$: Document
- $N$: Total number of documents
- $\text{DF}(t)$: Number of documents containing the term $t$


## Simplest embedding

In [ ]:
import torch
import torch.nn as nn

class CustomEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        """
        Custom implementation of an embedding layer.

        Args:
            num_embeddings (int): Size of the dictionary of embeddings (vocabulary size).
            embedding_dim (int): The size of each embedding vector.
        """
        super(CustomEmbedding, self).__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim

        # Initialize the embedding weights randomly
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))

    def forward(self, input_ids):
        """
        Forward pass for the embedding layer.

        Args:
            input_ids (torch.Tensor): Tensor of indices to look up in the embedding table.
                                     Shape: (batch_size, seq_len) or (seq_len,).

        Returns:
            torch.Tensor: Tensor of embeddings corresponding to the input indices.
                         Shape: (batch_size, seq_len, embedding_dim) or (seq_len, embedding_dim).
        """
        # Look up embeddings for the input indices
        return self.weight[input_ids]

# Example usage
if __name__ == "__main__":
    # Define parameters
    vocab_size = 10  # Number of words in the vocabulary
    embedding_dim = 5  # Dimension of each embedding vector
    batch_size = 2
    seq_len = 3

    # Create a custom embedding layer
    custom_embedding = CustomEmbedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    # Create input tensor (indices)
    input_ids = torch.tensor([[1, 2, 4], [0, 3, 9]])  # Shape: (batch_size, seq_len)

    # Forward pass
    embeddings = custom_embedding(input_ids)
    print("Input IDs:\n", input_ids)
    print("Embeddings:\n", embeddings)

Input IDs:
 tensor([[1, 2, 4],
        [0, 3, 9]])
Embeddings:
 tensor([[[ 0.8841,  0.2158,  0.9716,  1.0074, -0.1318],
         [-0.0682, -0.5498,  0.8141,  1.4693, -0.1168],
         [-0.4945,  1.9058,  1.5834, -1.9117, -0.4450]],

        [[-2.0951, -0.1727,  0.8738, -0.3310,  0.2838],
         [-0.0962, -1.0197, -0.9410,  0.1081, -0.0294],
         [ 0.7818,  0.0274,  0.7741,  0.0077, -0.8766]]],
       grad_fn=<IndexBackward0>)


## High level Emebedding model in Python instead of C++/ Cuda

Embeddings:
 tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.7988,  0.8778,  1.3247,  0.5557,  0.7264],
         [ 0.4432,  1.2291,  1.1268,  0.9214, -0.4176]],

        [[-0.1620, -0.7091,  1.2028, -0.6621, -0.9345],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [-1.8796,  0.2774, -0.6164, -0.0272, -0.0972]]])
Norms:
 tensor([[0.0000, 2.0000, 2.0000],
        [1.8131, 0.0000, 2.0000]])


In [ ]:
input = torch.randint(0,10,size=(2,3), dtype=torch.int32)
input_flat = input.reshape(-1)
print(input_flat)

weight = nn.Parameter(torch.randint(0,10, size=(10, 5), dtype = torch.float32))

embeddings_flat = torch.index_select(
            weight ,
            dim=0,
            index=input_flat
        )

print(weight)
print(embeddings_flat)

# Step 3: Reshape to original input shape + embedding dimension
output = embeddings_flat.reshape(*input.shape, 5)
print(output)

tensor([4, 9, 7, 3, 0, 8], dtype=torch.int32)
Parameter containing:
tensor([[8., 8., 4., 9., 5.],
        [8., 4., 8., 8., 2.],
        [4., 7., 6., 6., 0.],
        [5., 7., 5., 2., 1.],
        [0., 9., 1., 5., 8.],
        [7., 7., 6., 1., 1.],
        [4., 4., 5., 1., 3.],
        [1., 3., 7., 0., 4.],
        [4., 2., 7., 1., 3.],
        [5., 6., 7., 0., 2.]], requires_grad=True)
tensor([[0., 9., 1., 5., 8.],
        [5., 6., 7., 0., 2.],
        [1., 3., 7., 0., 4.],
        [5., 7., 5., 2., 1.],
        [8., 8., 4., 9., 5.],
        [4., 2., 7., 1., 3.]], grad_fn=<IndexSelectBackward0>)
tensor([[[0., 9., 1., 5., 8.],
         [5., 6., 7., 0., 2.],
         [1., 3., 7., 0., 4.]],

        [[5., 7., 5., 2., 1.],
         [8., 8., 4., 9., 5.],
         [4., 2., 7., 1., 3.]]], grad_fn=<ViewBackward0>)


## Word2Vec


Word2Vec is a neural network model used for natural language processing tasks to generate word embeddings—dense vector representations of words. Developed by Google, this model captures semantic meanings and relationships between words by training on large text corpora. It comes in two main architectures: Continuous Bag of Words (CBOW) and Skip-Gram. CBOW predicts a target word from its surrounding context, whereas Skip-Gram predicts context words given a target word. These embeddings can then be used in various NLP applications, such as text classification, clustering, and recommendation systems, where similar words are positioned closer in the vector space, reflecting their contextual similarity.

### SkimGram




#### SkimGram - Simple method ( No applicable)
https://www.youtube.com/watch?v=jKUwzgzdz3U&t=115s

**Skip-Gram** is a variant of the Word2Vec model used to learn word embeddings. Unlike the Continuous Bag of Words (CBOW) model, which predicts a target word given its surrounding context words, Skip-Gram does the opposite—it predicts the context words given a target word. In this approach, each word is treated as the central word, and the model attempts to predict words that appear within a predefined window size around it.

The main advantage of Skip-Gram is its effectiveness in capturing semantic relationships between words, especially when training on large datasets. By focusing on predicting context words, it can generate highly accurate and meaningful embeddings that can be used in various NLP applications.



**1. One-Hot Encoding of the Input Word**

The center word is represented as a one-hot vector of dimension $V$ (vocabulary size):

$$
\mathbf{x} = \begin{bmatrix} 0 \\ 1 \\ 0 \\ \vdots \\ 0 \end{bmatrix} \in \mathbb{R}^{V}
$$

**2. Embedding Lookup**

The input embedding matrix is defined as $ \mathbf{W}_{\text{in}} \in \mathbb{R}^{V \times D} $, where $D$ is the embedding dimension. The embedding for the center word is obtained by:

$$
\mathbf{h} = \mathbf{x}^\top \mathbf{W}_{\text{in}}
$$

Thus, $ \mathbf{h} \in \mathbb{R}^{D} $.

**3. Output Layer (Logits Computation)**

The output embedding matrix is given by $ \mathbf{W}_{\text{out}} \in \mathbb{R}^{D \times V} $. The logits (unnormalized scores) for each word in the vocabulary are computed as:

$$
\mathbf{z} = \mathbf{h} \, \mathbf{W}_{\text{out}}
$$

where $ \mathbf{z} \in \mathbb{R}^{V} $. Each element is computed by:

$$
z_i = \mathbf{h}^\top \mathbf{v}_i
$$

with $ \mathbf{v}_i $ being the $i^\text{th}$ column of $ \mathbf{W}_{\text{out}} $.

**4. Softmax Function**

The softmax function converts the logits into a probability distribution over the vocabulary:

$$
\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}} \quad \text{for } i=1,2,\ldots,V
$$

So the resulting vector is $ \hat{\mathbf{y}} \in \mathbb{R}^{V} $.

**5. Skip-Gram Conditional Probability**

The probability of a context word $w_O$ given a center word $w_I$ is defined as:

$$
p(w_O \mid w_I) = \frac{\exp\left(\mathbf{u}_{w_I}^\top \mathbf{v}_{w_O}\right)}{\sum_{w=1}^{V} \exp\left(\mathbf{u}_{w_I}^\top \mathbf{v}_w\right)}
$$

where $ \mathbf{u}_{w_I} $ is the input (center) word embedding, and $ \mathbf{v}_{w_O} $ is the output (context) word embedding.

**6. Loss Function**

For a center word $w_I$ with $C$ context words $\{w_{O_1}, w_{O_2}, \ldots, w_{O_C}\}$, the loss is the negative log-likelihood:

$$
L = -\sum_{c=1}^{C} \log p(w_{O_c} \mid w_I)
$$

Explicitly, substituting the conditional probability:

$$
L = -\sum_{c=1}^{C} \log \left( \frac{\exp\left(\mathbf{u}_{w_I}^\top \mathbf{v}_{w_{O_c}}\right)}{\sum_{w=1}^{V} \exp\left(\mathbf{u}_{w_I}^\top \mathbf{v}_w\right)} \right)
$$



In [ ]:
# This method is extremly ineffictive as we have to get probabilities for word in the data for each word. for large dataset it is impossible
# Just for learning only
# use skim-gram with Negative sampling.

import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader

class SimpleWord2VecDataset(Dataset):
    def __init__(self, text, window_size=2):
        self.text = text
        self.window_size = window_size

        # Build vocabulary
        self.vocab = list(set(text))
        self.word2idx = {word: idx for idx, word in enumerate(self.vocab)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        self.vocab_size = len(self.vocab)

        # Generate training pairs
        self.data = []
        for i, target_word in enumerate(text):
            start = max(0, i - window_size)
            end = min(len(text), i + window_size + 1)
            context_words = text[start:i] + text[i+1:end]

            for context_word in context_words:
                self.data.append((
                    self.word2idx[target_word],
                    self.word2idx[context_word]
                ))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

class SimpleWord2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Single embedding layer for input words
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Linear layer for output predictions
        self.fc = nn.Linear(embedding_dim, vocab_size)

        # Initialize embeddings
        nn.init.xavier_uniform_(self.embeddings.weight)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, target):
        """
        Args:
            target: Tensor of target word indices (batch_size,)
        Returns:
            logits: Prediction scores for context words (batch_size, vocab_size)
        """
        emb = self.embeddings(target)  # (batch_size, embedding_dim)
        logits = self.fc(emb)          # (batch_size, vocab_size)
        return logits

# Training setup
text = "the quick brown fox jumps over the lazy dog".split()
dataset = SimpleWord2VecDataset(text, window_size=2)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

model = SimpleWord2Vec(dataset.vocab_size, embedding_dim=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(100):
    total_loss = 0
    for targets, contexts in dataloader:
        optimizer.zero_grad()

        # Forward pass
        logits = model(targets)
        # Calculate loss (predict context words)
        loss = criterion(logits, contexts)

        # Backpropagation
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

# Get word embeddings
word_vectors = model.embeddings.weight.data
print("\nLearned embeddings:")
for word, idx in dataset.word2idx.items():
    print(f"{word}: {word_vectors[idx][:5].tolist()}...")

Epoch 1, Loss: 2.1405
Epoch 2, Loss: 2.0519
Epoch 3, Loss: 2.0405
Epoch 4, Loss: 1.9562
Epoch 5, Loss: 1.9395
Epoch 6, Loss: 1.9057
Epoch 7, Loss: 1.8758
Epoch 8, Loss: 1.8389
Epoch 9, Loss: 1.8193
Epoch 10, Loss: 1.7691
Epoch 11, Loss: 1.7650
Epoch 12, Loss: 1.7801
Epoch 13, Loss: 1.7289
Epoch 14, Loss: 1.6981
Epoch 15, Loss: 1.6900
Epoch 16, Loss: 1.6845
Epoch 17, Loss: 1.6581
Epoch 18, Loss: 1.6045
Epoch 19, Loss: 1.5775
Epoch 20, Loss: 1.5759
Epoch 21, Loss: 1.5409
Epoch 22, Loss: 1.5414
Epoch 23, Loss: 1.5568
Epoch 24, Loss: 1.5158
Epoch 25, Loss: 1.5130
Epoch 26, Loss: 1.5314
Epoch 27, Loss: 1.5074
Epoch 28, Loss: 1.5106
Epoch 29, Loss: 1.4839
Epoch 30, Loss: 1.4875
Epoch 31, Loss: 1.4597
Epoch 32, Loss: 1.4798
Epoch 33, Loss: 1.4624
Epoch 34, Loss: 1.4484
Epoch 35, Loss: 1.4489
Epoch 36, Loss: 1.4660
Epoch 37, Loss: 1.4524
Epoch 38, Loss: 1.4411
Epoch 39, Loss: 1.4516
Epoch 40, Loss: 1.4655
Epoch 41, Loss: 1.4530
Epoch 42, Loss: 1.4135
Epoch 43, Loss: 1.4446
Epoch 44, Loss: 1.42

#### SkimGram with Negative sampling

https://www.youtube.com/watch?v=CjCFJAGZEio&t=652s


The Skip-Gram model with Negative Sampling aims to maximize the likelihood of observed (positive) target-context pairs while minimizing the likelihood of randomly drawn (negative) pairs.

For a given target word $w_I$, a positive context word $w_O$, and $K$ negative samples $\{w_{N_1}, w_{N_2}, \dots, w_{N_K}\}$, the objective function for a single training instance is defined as:

$$
J = \log \sigma\left(\mathbf{u}_{w_O}^\top \mathbf{v}_{w_I}\right) + \sum_{i=1}^{K} \log \sigma\left(-\mathbf{u}_{w_{N_i}}^\top \mathbf{v}_{w_I}\right)
$$

where:
- $\mathbf{v}_{w_I}$ is the embedding of the target word,
- $\mathbf{u}_{w_O}$ is the embedding of the positive context word,
- $\mathbf{u}_{w_{N_i}}$ are the embeddings of the negative samples,
- and $\sigma(x)= \frac{1}{1+e^{-x}}$ is the sigmoid function.

The corresponding loss (to be minimized) is the negative of the objective:

$$
L = -J = -\left[\log \sigma\left(\mathbf{u}_{w_O}^\top \mathbf{v}_{w_I}\right) + \sum_{i=1}^{K} \log \sigma\left(-\mathbf{u}_{w_{N_i}}^\top \mathbf{v}_{w_I}\right)\right]
$$

The training process minimizes the average loss over all training pairs to learn meaningful word representations.


In [ ]:
"""
Word2Vec (Skip-Gram with Negative Sampling) Implementation from Scratch
Reference: "Distributed Representations of Words and Phrases and their Compositionality" (Mikolov et al., 2013)
"""

"""
Negative sampling is a technique used to optimize the training process of models like Word2Vec, particularly the Skip-Gram model. Instead of calculating the probabilities of all words in the vocabulary, which can be computationally expensive, negative sampling simplifies this process.

In negative sampling, for each target-context word pair, a few "negative" word pairs are generated. These negative pairs are random words that don't co-occur with the target word. The model then updates its weights by making the target-context pairs more likely and the negative pairs less likely. This approach significantly reduces the computational load and speeds up the training process while still learning meaningful word representations.

Here's a practical example:
- **Positive Pair (target-context):** "cat" - "meow"
- **Negative Pairs (random):** "cat" - "car", "cat" - "tree"

By focusing on these pairs, the model efficiently learns word embeddings without having to consider the entire vocabulary. This technique is widely used in natural language processing to make models more scalable and efficient.

Here's a helpful [link](https://rare-technologies.com/word2vec-tutorial/#negative-sampling) for more details. Let me know if you'd like further explanations!
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader

# ======================================================================
#                         DATA PREPARATION
# ======================================================================

class Word2VecDataset(Dataset):
    """
    Custom Dataset class for Word2Vec training data generation
    Implements:
    - Vocabulary building
    - Training pair generation (target-context pairs)
    - Negative sampling
    """

    def __init__(self, text, window_size=2, num_neg_samples=5):
        """
        Args:
            text: List of words from the corpus
            window_size: Context window radius (total window size = 2*window_size+1)
            num_neg_samples: Number of negative samples per positive pair
        """
        self.text = text
        self.window_size = window_size
        self.num_neg_samples = num_neg_samples

        # Build vocabulary and mappings
        self.vocab, self.word2idx, self.idx2word = self.build_vocab()
        self.vocab_size = len(self.vocab)

        # Generate training data (target-context pairs)
        self.data = self.generate_training_data()

        # Calculate word counts # added this line
        self.word_counts = Counter(self.text) # added this line


        # Create noise distribution for negative sampling
        # Using P(w)^(3/4) as suggested in the original paper
        word_freq = np.array([self.word_counts[w] for w in self.vocab])
        self.noise_dist = word_freq ** 0.75  # Diminish frequency imbalance for high occuring word like the, he
        self.noise_dist /= self.noise_dist.sum()  # Normalize to probability distribution

    def build_vocab(self):
        """Create vocabulary and word<->index mappings"""
        vocab = list(set(self.text))  # Get unique words
        word2idx = {word: idx for idx, word in enumerate(vocab)}
        idx2word = {idx: word for word, idx in word2idx.items()}
        return vocab, word2idx, idx2word

    def generate_training_data(self):
        """Generate (target, context) pairs using sliding window"""
        """
        The data will be something like:
        (The, quick), (The, brown),
        (quick, The), (quick, brown), (quick, fox),
        (brown, The), (brown, quick), (brown, fox), (brown, jumps),
        (fox, quick), (fox, brown), (fox, jumps), (fox, over),
        (jumps, brown), (jumps, fox), (jumps, over), (jumps, the),
        (over, fox), (over, jumps), (over, the), (over, lazy),
        (the, jumps), (the, over), (the, lazy), (the, dog),
        (lazy, over), (lazy, the), (lazy, dog),
        (dog, the), (dog, lazy)
        """
        data = []
        for i, target_word in enumerate(self.text):
            # Get context words within window (excluding target word itself)
            start = max(0, i - self.window_size)
            end = min(len(self.text), i + self.window_size + 1)
            context_words = self.text[start:i] + self.text[i+1:end]

            # Create pairs for all context words
            for context_word in context_words:
                data.append((self.word2idx[target_word],
                           self.word2idx[context_word]))
        return data

    def __len__(self):
        """Return total number of training pairs"""
        return len(self.data)

    def __getitem__(self, idx):
        """
        Return a training sample with negative sampling
        Returns:
            - target: Target word index
            - context: Positive context word index
            - neg_context: List of negative sample indices
        """
        target, context = self.data[idx]

        # Generate negative samples from noise distribution
        # To improve later in future. make sure the neg_context is different than positive context
        neg_samples = np.random.choice(
            self.vocab_size,
            size=self.num_neg_samples,
            p=self.noise_dist
        )
        return target, context, neg_samples

# ======================================================================
#                         MODEL DEFINITION
# ======================================================================

class Word2Vec(nn.Module):
    """
    Skip-Gram with Negative Sampling (SGNS) model
    Architecture:
    - Two embedding layers: input (target) and output (context) embeddings
    - Loss: Negative sampling loss comparing positive vs negative pairs
    """

    def __init__(self, vocab_size, embedding_dim):
        """
        Args:
            vocab_size: Size of vocabulary
            embedding_dim: Dimension of word embeddings
        """
        super().__init__()
        # Input embeddings (target words)
        self.input_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Output embeddings (context words)
        self.output_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Initialize embeddings using Xavier uniform distribution
        nn.init.xavier_uniform_(self.input_embeddings.weight)
        nn.init.xavier_uniform_(self.output_embeddings.weight)

    def forward(self, target, context, neg_context):
        """
        Forward pass with loss calculation
        Args:
            target: Tensor of target word indices (batch_size,)
            context: Tensor of positive context indices (batch_size,)
            neg_context: Tensor of negative context indices (batch_size, num_neg)
        Returns:
            loss: Computed negative sampling loss
        """
        # Get embeddings for all words
        # Shape: (batch_size, embedding_dim)
        target_emb = self.input_embeddings(target)
        context_emb = self.output_embeddings(context)
        # Shape: (batch_size, num_neg, embedding_dim)
        neg_emb = self.output_embeddings(neg_context)

        # Calculate positive similarity scores
        # we would have done dot product like target_emb.T @ context_emb like in formula
        # But we have batch as first dimension.
        # But we could have done for loop ( target_emb[0].T @ context_emb[0], for 0, 1,2,...batch_size)
        # But below way also do like below as we going to get single answer for each batch
        # element wise product between target and positive context embeddings
        pos_scores = torch.sum(context_emb * target_emb, dim=1)  # (batch_size,)
        # we want torch.sigmoid(pos_scores) closer to 1 for positive context
        pos_loss = torch.log(torch.sigmoid(pos_scores))  # log(σ(s))

        # Calculate negative similarity scores
        # Batch matrix multiplication: (batch_size, num_neg, emb_dim) * (batch_size, emb_dim, 1)
        # Result: (batch_size, num_neg, 1) -> squeeze to (batch_size, num_neg)
        neg_scores = torch.bmm(neg_emb, target_emb.unsqueeze(2)).squeeze()
        neg_loss = torch.sum(torch.log(torch.sigmoid(-neg_scores)), dim=1)  # log(σ(-s))

        # Combine losses and return mean negative log likelihood
        return -(pos_loss + neg_loss).mean()

# ======================================================================
#                         TRAINING SETUP
# ======================================================================

# Sample text corpus (can be replaced with any text data)
text = [
    "the quick brown fox jumps over the lazy dog",
    "i love machine learning and deep learning"
]
# Flatten into list of words
text = [sentence.split() for sentence in text]
text = [word for sentence in text for word in sentence]

# Hyperparameters
EMBEDDING_DIM = 100    # Dimension of word vectors (typically 100-300)
BATCH_SIZE = 32        # Number of samples per batch
WINDOW_SIZE = 2        # Context window radius
NUM_NEG_SAMPLES = 5    # Negative samples per positive pair
EPOCHS = 50            # Number of training epochs
LEARNING_RATE = 0.001  # Learning rate for optimizer

# Create dataset and dataloader
dataset = Word2VecDataset(text, WINDOW_SIZE, NUM_NEG_SAMPLES)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model, optimizer
model = Word2Vec(dataset.vocab_size, EMBEDDING_DIM)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Adam optimizer for adaptive learning

# ======================================================================
#                         TRAINING LOOP
# ======================================================================

print("Starting training...")
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_idx, batch in enumerate(dataloader):
        # Unpack batch data
        target, context, neg_context = batch

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass and loss calculation
        loss = model(target, context, neg_context)

        # Backpropagation and optimization
        loss.backward()
        optimizer.step()

        # Accumulate loss for monitoring
        total_loss += loss.item()

    # Print epoch statistics
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} \t Loss: {avg_loss:.4f}")

# ======================================================================
#                         EMBEDDING EXTRACTION
# ======================================================================

def get_word_vectors(model, word2idx):
    """
    Extract learned word embeddings from the model
    Args:
        model: Trained Word2Vec model
        word2idx: Vocabulary mapping
    Returns:
        Dictionary mapping words to their embeddings
    """
    # Use input embeddings as final word vectors
    embeddings = model.input_embeddings.weight.data
    return {word: embeddings[idx] for word, idx in word2idx.items()}

# Get word vectors and show examples
word_vectors = get_word_vectors(model, dataset.word2idx)

print("\nSample word vectors:")
for word in ["quick", "machine", "learning"]:
    if word in word_vectors:
        print(f"{word}: {word_vectors[word][:5]}...")  # Show first 5 dimensions

# ======================================================================
#                         HOW TO USE EMBEDDINGS
# ======================================================================
"""
The learned embeddings can be used for:
1. Semantic similarity: word_vectors["king"] @ word_vectors["queen"].T
2. As input to other NLP models
3. Analogies: vec("king") - vec("man") + vec("woman") ≈ vec("queen")
4. Visualization with t-SNE/PCA
"""

Starting training...
Epoch 1/50 	 Loss: 4.2197
Epoch 2/50 	 Loss: 4.1709
Epoch 3/50 	 Loss: 4.1546
Epoch 4/50 	 Loss: 4.0797
Epoch 5/50 	 Loss: 4.1188
Epoch 6/50 	 Loss: 4.0841
Epoch 7/50 	 Loss: 4.1001
Epoch 8/50 	 Loss: 4.0445
Epoch 9/50 	 Loss: 4.0572
Epoch 10/50 	 Loss: 4.0188
Epoch 11/50 	 Loss: 4.0606
Epoch 12/50 	 Loss: 4.0001
Epoch 13/50 	 Loss: 3.9969
Epoch 14/50 	 Loss: 3.9437
Epoch 15/50 	 Loss: 3.9409
Epoch 16/50 	 Loss: 3.9207
Epoch 17/50 	 Loss: 3.9320
Epoch 18/50 	 Loss: 3.9458
Epoch 19/50 	 Loss: 3.8843
Epoch 20/50 	 Loss: 3.8451
Epoch 21/50 	 Loss: 3.8903
Epoch 22/50 	 Loss: 3.8709
Epoch 23/50 	 Loss: 3.8321
Epoch 24/50 	 Loss: 3.8301
Epoch 25/50 	 Loss: 3.8090
Epoch 26/50 	 Loss: 3.7369
Epoch 27/50 	 Loss: 3.7718
Epoch 28/50 	 Loss: 3.6997
Epoch 29/50 	 Loss: 3.6974
Epoch 30/50 	 Loss: 3.7244
Epoch 31/50 	 Loss: 3.7045
Epoch 32/50 	 Loss: 3.6136
Epoch 33/50 	 Loss: 3.6024
Epoch 34/50 	 Loss: 3.5703
Epoch 35/50 	 Loss: 3.6040
Epoch 36/50 	 Loss: 3.5608
Epoch 37/50 	 Lo

'\nThe learned embeddings can be used for:\n1. Semantic similarity: word_vectors["king"] @ word_vectors["queen"].T\n2. As input to other NLP models\n3. Analogies: vec("king") - vec("man") + vec("woman") ≈ vec("queen")\n4. Visualization with t-SNE/PCA\n'

### CBOW


#### CBOW- simple Method (Not applicable in real word)
https://www.youtube.com/watch?v=ZcHfDEYOPIQ


The CBOW model predicts a target word given its surrounding context words. In this implementation, we use a fixed window size, where the context is defined as the $w$ words before and after the target word.

## 1. Data Preparation and Training Pairs

Given a text sequence, for each target word at position $i$, the context is defined as:

$$
\text{Context} = \{w_{i-w}, \dots, w_{i-1}, w_{i+1}, \dots, w_{i+w}\}
$$

where $w$ is the window size.

Each word is converted to an index based on a vocabulary, and training pairs are generated as:

$$
(\text{context indices}, \text{target index})
$$

## 2. Model Architecture

The CBOW model uses:
- **An embedding layer:** Each word index is mapped to a vector $u \in \mathbb{R}^{d}$.
- **Context vector computation:** The embeddings for the context words are averaged:
  
$$
v_C = \frac{1}{N} \sum_{j=1}^{N} u_{w_{C_j}}
$$

where $N = 2 \times \text{window size}$ (i.e. the total number of context words).

- **A linear (fully-connected) layer:** The averaged context vector is transformed into scores (logits) for each word in the vocabulary using a weight matrix $W \in \mathbb{R}^{d \times V}$ (and bias $b \in \mathbb{R}^{V}$):

$$
z = W^\top v_C + b
$$

- **Prediction:** The logits are then passed through a softmax function to obtain probabilities for each word:

$$
P(w \mid \text{Context}) = \frac{\exp(z_w)}{\sum_{j=1}^{V} \exp(z_j)}
$$

## 3. Loss Function

The model is trained using the cross-entropy loss. For a given target word $w_t$, the loss is:

$$
\mathcal{L} = -\log P(w_t \mid \text{Context})
$$

The overall training loss is averaged over all training samples.

---

## Code-to-Math Correspondence

The following table maps key code components to their mathematical representations:

$$
\begin{array}{|l|l|}
\hline
\textbf{Code Component} & \textbf{Mathematical Representation} \\
\hline
\mathtt{embeds = self.embeddings(context)} & \text{Embeddings for each context word } u_{w_{C_j}}, \text{ shape } (B, N, d) \\
\hline
\mathtt{avg\_embeds = torch.mean(embeds,\, dim=1)} & v_C = \frac{1}{N}\sum_{j=1}^{N} u_{w_{C_j}}, \text{ shape } (B, d) \\
\hline
\mathtt{logits = self.fc(avg\_embeds)} & z = W^\top v_C + b, \text{ shape } (B, V) \\
\hline
\mathtt{loss = criterion(logits, targets)} & \mathcal{L} = -\log P(w_t \mid \text{Context}) \\
\hline
\end{array}
$$

where:
- $B$ is the batch size,
- $N$ is the number of context words (typically $2 \times \text{window size}$),
- $d$ is the embedding dimension,
- $V$ is the vocabulary size.

---

This explanation summarizes the key mathematical operations of the CBOW model and shows how they correspond to the implementation code.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ======================================================================
#                         DATA PREPARATION
# ======================================================================

class CBOWDataset(Dataset):
    """
    Custom Dataset class for CBOW training data generation
    Generates (context, target) pairs with fixed window size
    """

    def __init__(self, text, window_size=2):
        """
        Args:
            text: List of words from the corpus
            window_size: Number of words to consider on each side of target
        """
        self.text = text
        self.window_size = window_size

        # Build vocabulary and mappings
        self.vocab = list(set(text))
        self.word2idx = {word: idx for idx, word in enumerate(self.vocab)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        self.vocab_size = len(self.vocab)

        # Generate training data (context, target pairs)
        self.data = self.generate_training_data()

    def generate_training_data(self):
        """Create (context, target) pairs with fixed window size"""
        data = []

        # Iterate through possible target positions
        for i in range(self.window_size, len(self.text) - self.window_size):
            # Get target word at center
            target = self.text[i]

            # Get context words (window_size words on each side)
            context = (
                self.text[i - self.window_size : i] +  # Left context
                self.text[i + 1 : i + self.window_size + 1]  # Right context
            )

            # Store as indices
            data.append((
                [self.word2idx[w] for w in context],  # Context indices
                self.word2idx[target]                  # Target index
            ))
        return data

    def __len__(self):
        """Return total number of training pairs"""
        return len(self.data)

    def __getitem__(self, idx):
        """
        Return a training sample
        Returns:
            context: Tensor of context word indices
            target: Tensor of target word index
        """
        context, target = self.data[idx]
        return torch.tensor(context), torch.tensor(target)

# ======================================================================
#                         MODEL DEFINITION
# ======================================================================

class CBOW(nn.Module):
    """
    Continuous Bag of Words (CBOW) model
    Architecture:
    - Embedding layer converts context indices to vectors
    - Average of context embeddings is used to predict target word
    """

    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Embedding layer (shared for all context words)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Final linear layer for prediction
        self.fc = nn.Linear(embedding_dim, vocab_size)

        # Initialize weights
        nn.init.xavier_uniform_(self.embeddings.weight)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, context):
        """
        Forward pass
        Args:
            context: Tensor of context indices (batch_size, window_size*2)
        Returns:
            logits: Prediction scores for target word (batch_size, vocab_size)
        """
        # Get embeddings for all context words
        # Shape: (batch_size, context_size, embedding_dim)
        embeds = self.embeddings(context)

        # Average the embeddings across context words
        # Shape: (batch_size, embedding_dim)
        avg_embeds = torch.mean(embeds, dim=1)

        # Convert to vocabulary scores
        # Shape: (batch_size, vocab_size)
        logits = self.fc(avg_embeds)
        return logits

# ======================================================================
#                         TRAINING SETUP
# ======================================================================

# Sample text corpus
text = "the quick brown fox jumps over the lazy dog".split()

# Hyperparameters
EMBEDDING_DIM = 10    # Dimension of word vectors
WINDOW_SIZE = 2       # Context window radius (total context words = 2*WINDOW_SIZE)
BATCH_SIZE = 2        # Number of samples per batch
EPOCHS = 100          # Number of training epochs
LEARNING_RATE = 0.01  # Learning rate for optimizer

# Create dataset and dataloader
dataset = CBOWDataset(text, WINDOW_SIZE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model, loss, and optimizer
model = CBOW(dataset.vocab_size, EMBEDDING_DIM)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ======================================================================
#                         TRAINING LOOP
# ======================================================================

print("Starting training...")
for epoch in range(EPOCHS):
    total_loss = 0
    for contexts, targets in dataloader:
        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(contexts)

        # Calculate loss
        loss = criterion(logits, targets)

        # Backpropagation
        loss.backward()
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

    # Print progress
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} \t Loss: {avg_loss:.4f}")

# ======================================================================
#                         EMBEDDING EXTRACTION
# ======================================================================

def get_word_vectors(model, word2idx):
    """Extract learned word embeddings"""
    return {word: model.embeddings.weight.data[idx]
            for word, idx in word2idx.items()}

word_vectors = get_word_vectors(model, dataset.word2idx)

print("\nLearned embeddings:")
for word in ["quick", "fox", "lazy"]:
    print(f"{word}: {word_vectors[word][:5].tolist()}...")

# ======================================================================
#                         SIMILARITY EXAMPLE
# ======================================================================

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    return torch.dot(vec1, vec2) / (torch.norm(vec1) * torch.norm(vec2))

# Find most similar words to "fox"
target_word = "fox"
target_vector = word_vectors[target_word]

similarities = {
    word: cosine_similarity(target_vector, vec)
    for word, vec in word_vectors.items()
}

print(f"\nWords most similar to '{target_word}':")
for word, score in sorted(similarities.items(), key=lambda x: -x[1]):
    print(f"{word}: {score:.3f}")

Starting training...
Epoch 1/100 	 Loss: 2.1273
Epoch 2/100 	 Loss: 1.9873
Epoch 3/100 	 Loss: 1.9305
Epoch 4/100 	 Loss: 1.9011
Epoch 5/100 	 Loss: 1.8545
Epoch 6/100 	 Loss: 1.7780
Epoch 7/100 	 Loss: 1.7308
Epoch 8/100 	 Loss: 1.6854
Epoch 9/100 	 Loss: 1.6379
Epoch 10/100 	 Loss: 1.6681
Epoch 11/100 	 Loss: 1.6183
Epoch 12/100 	 Loss: 1.4834
Epoch 13/100 	 Loss: 1.5171
Epoch 14/100 	 Loss: 1.4098
Epoch 15/100 	 Loss: 1.3611
Epoch 16/100 	 Loss: 1.3119
Epoch 17/100 	 Loss: 1.2188
Epoch 18/100 	 Loss: 1.2125
Epoch 19/100 	 Loss: 1.1123
Epoch 20/100 	 Loss: 1.1317
Epoch 21/100 	 Loss: 1.0642
Epoch 22/100 	 Loss: 0.9554
Epoch 23/100 	 Loss: 1.0244
Epoch 24/100 	 Loss: 0.9390
Epoch 25/100 	 Loss: 0.9062
Epoch 26/100 	 Loss: 0.8823
Epoch 27/100 	 Loss: 0.7745
Epoch 28/100 	 Loss: 0.7772
Epoch 29/100 	 Loss: 0.6894
Epoch 30/100 	 Loss: 0.6992
Epoch 31/100 	 Loss: 0.6098
Epoch 32/100 	 Loss: 0.5500
Epoch 33/100 	 Loss: 0.5951
Epoch 34/100 	 Loss: 0.5752
Epoch 35/100 	 Loss: 0.5340
Epoch 36

#### CBOW - with Negative Sampling


The **CBOW model** predicts a target word $w_I$ given its surrounding **context words** $\{w_{C_1}, w_{C_2}, \dots, w_{C_m}\}$, where $m$ is the number of context words.

## 1. Compute the Context Vector
Each context word $w_{C_j}$ has an **input embedding** $u_{w_{C_j}}$.  
The **context vector** is computed as the average of the embeddings of all context words:

$$
v_C = \frac{1}{m} \sum_{j=1}^{m} u_{w_{C_j}}
$$

where:
- $m$ is the number of context words.
- $v_C$ is the average context embedding.

## 2. Compute Positive Score (Target Word Prediction)
The **target word** $w_I$ has an **output embedding** $v_{w_I}$.  
The model predicts $w_I$ by computing the **dot product** of the context vector and the target word's output embedding:

$$
s^+ = v_C^\top v_{w_I}
$$

The probability of $w_I$ given $v_C$ is:

$$
P(w_I \mid C) = \sigma(s^+) = \frac{1}{1+e^{-s^+}}
$$

## 3. Compute Negative Scores
For **negative sampling**, we sample $K$ **negative words** $\{w_{N_1}, w_{N_2}, \dots, w_{N_K}\}$ that are **not in the actual context**.  
Each negative word $w_{N_k}$ has an output embedding $v_{w_{N_k}}$, and its score is computed as:

$$
s^-_k = v_C^\top v_{w_{N_k}}
$$

The probability of **not** predicting a negative sample is:

$$
P(w_{N_k} \mid C) = \sigma(-s^-_k) = \frac{1}{1+e^{s^-_k}}
$$

## 4. Compute Loss Function
The **objective function** for one training sample is:

$$
J = \log \sigma(s^+) + \sum_{k=1}^{K} \log \sigma(-s^-_k)
$$

To train the model, we **minimize** the **negative log likelihood**:

$$
L = -J = -\left(\log \sigma(s^+) + \sum_{k=1}^{K} \log \sigma(-s^-_k)\right)
$$

This loss is **averaged** over all training samples.

---

## Code to Math Correspondence
The following table maps code variables to their mathematical representations.

$$
\begin{array}{|l|l|}
\hline
\textbf{Code Component} & \textbf{Mathematical Representation} \\
\hline
\mathtt{context\_embeddings = self.in\_embeddings(context)} & u_{w_{C_j}} \text{ for each context word} \\
\hline
\mathtt{ctx\_embeds = context\_embeddings.mean(dim=1)} & v_C = \frac{1}{m} \sum_{j=1}^{m} u_{w_{C_j}} \\
\hline
\mathtt{target\_embeds = self.out\_embeddings(target)} & v_{w_I} \\
\hline
\mathtt{pos\_scores = torch.sum(ctx\_embeds * target\_embeds, \, dim=1)} & s^+ = v_C^\top v_{w_I} \\
\hline
\mathtt{neg\_embeds = self.out\_embeddings(neg\_samples)} & v_{w_{N_k}} \text{ for negative samples} \\
\hline
\mathtt{neg\_scores = torch.bmm(neg\_embeds, \, ctx\_embeds.unsqueeze(2)).squeeze(-1)} & s^-_k = v_C^\top v_{w_{N_k}} \\
\hline
\mathtt{pos\_loss = torch.log(torch.sigmoid(pos\_scores))} & \log \sigma(s^+) \\
\hline
\mathtt{neg\_loss = torch.sum(torch.log(torch.sigmoid(-neg\_scores)), \, dim=1)} & \sum_{k=1}^{K} \log \sigma(-s^-_k) \\
\hline
\mathtt{return \, -(pos\_loss + neg\_loss).mean()} & L = -\left(\log \sigma(s^+) + \sum_{k=1}^{K} \log \sigma(-s^-_k)\right) \\
\hline
\end{array}
$$


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader

# ======================================================================
#                         DATA PREPARATION
# ======================================================================

class CBOWNegativeSamplingDataset(Dataset):
    def __init__(self, text, window_size=2, num_neg_samples=5):
        self.text = text
        self.window_size = window_size
        self.num_neg_samples = num_neg_samples

        # Build vocabulary
        self.vocab, self.word2idx, self.idx2word = self.build_vocab()
        self.vocab_size = len(self.vocab)
        self.word_counts = Counter(text)

        # Generate training data
        self.data = self.generate_training_data()

        # Create noise distribution (unigram^0.75)
        word_freq = np.array([self.word_counts[w] for w in self.vocab])
        self.noise_dist = word_freq ** 0.75
        self.noise_dist /= self.noise_dist.sum()

    def build_vocab(self):
        vocab = list(set(self.text))
        word2idx = {word: idx for idx, word in enumerate(vocab)}
        idx2word = {idx: word for word, idx in word2idx.items()}
        return vocab, word2idx, idx2word

    def generate_training_data(self):
        data = []
        for i in range(self.window_size, len(self.text) - self.window_size):
            target = self.text[i]
            context = (
                self.text[i - self.window_size : i] +
                self.text[i + 1 : i + self.window_size + 1]
            )
            data.append((
                [self.word2idx[w] for w in context],
                self.word2idx[target]
            ))
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        neg_samples = np.random.choice(
            self.vocab_size,
            size=self.num_neg_samples,
            p=self.noise_dist
        )
        return torch.tensor(context), torch.tensor(target), torch.tensor(neg_samples)

# ======================================================================
#                         MODEL DEFINITION
# ======================================================================

class CBOWNegativeSampling(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Input embeddings (context words)
        self.in_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Output embeddings (target words)
        self.out_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Initialize weights
        nn.init.xavier_uniform_(self.in_embeddings.weight)
        nn.init.xavier_uniform_(self.out_embeddings.weight)
    def forward(self, context, target, neg_samples):
        """
        Args:
            context: (batch_size, num_context_words)
                    e.g. (2, 4) for batch_size=2, window_size=2
            target: (batch_size,)
                    e.g. (2,)
            neg_samples: (batch_size, num_neg_samples)
                        e.g. (2, 5)
        """
        # Embedding lookup for context words
        # Shape: (batch_size, num_context_words, embedding_dim)
        # e.g. (2, 4, 100)
        context_embeddings = self.in_embeddings(context)

        # Average across context words dimension
        # Shape: (batch_size, embedding_dim)
        # e.g. (2, 100)
        ctx_embeds = context_embeddings.mean(dim=1)

        # Positive sample embedding lookup
        # Shape: (batch_size, embedding_dim)
        # e.g. (2, 100)
        target_embeds = self.out_embeddings(target)

        # Calculate positive scores (dot product)
        # Shape: (batch_size,)
        # e.g. (2,)
        pos_scores = torch.sum(ctx_embeds * target_embeds, dim=1)

        # Negative samples embedding lookup
        # Shape: (batch_size, num_neg_samples, embedding_dim)
        # e.g. (2, 5, 100)
        neg_embeds = self.out_embeddings(neg_samples)

        # Calculate negative scores (batch matrix multiplication)
        # ctx_embeds.unsqueeze(2) shape: (batch_size, embedding_dim, 1)
        # Result of bmm: (batch_size, num_neg_samples, 1)
        # After squeeze(-1): (batch_size, num_neg_samples)
        # e.g. (2, 5)
        neg_scores = torch.bmm(neg_embeds, ctx_embeds.unsqueeze(2)).squeeze(-1)

        # Calculate losses
        pos_loss = torch.log(torch.sigmoid(pos_scores))  # (batch_size,)
        neg_loss = torch.sum(torch.log(torch.sigmoid(-neg_scores)), dim=1)  # (batch_size,)

        # Final loss
        return -(pos_loss + neg_loss).mean()  # scalar

# ======================================================================
#                         TRAINING SETUP
# ======================================================================

# Sample corpus
text = "the quick brown fox jumps over the lazy dog".split()

# Hyperparameters
EMBEDDING_DIM = 50
WINDOW_SIZE = 2
NUM_NEG_SAMPLES = 5
BATCH_SIZE = 2
EPOCHS = 200
LEARNING_RATE = 0.025

# Initialize dataset and dataloader
dataset = CBOWNegativeSamplingDataset(text, WINDOW_SIZE, NUM_NEG_SAMPLES)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model and optimizer
model = CBOWNegativeSampling(dataset.vocab_size, EMBEDDING_DIM)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ======================================================================
#                         TRAINING LOOP
# ======================================================================

print("Starting training...")
for epoch in range(EPOCHS):
    total_loss = 0
    for contexts, targets, neg_samples in dataloader:
        optimizer.zero_grad()

        # Forward pass
        loss = model(contexts, targets, neg_samples)

        # Backpropagation
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} \t Loss: {avg_loss:.4f}")

# ======================================================================
#                         EMBEDDING UTILITIES
# ======================================================================

def get_word_vectors(model, word2idx):
    """Extract input embeddings as final word vectors"""
    return {word: model.in_embeddings.weight.data[idx]
            for word, idx in word2idx.items()}

word_vectors = get_word_vectors(model, dataset.word2idx)

# Example similarity check
def cosine_similarity(vec1, vec2):
    return torch.dot(vec1, vec2) / (torch.norm(vec1) * torch.norm(vec2))

print("\nMost similar to 'fox':")
fox_vec = word_vectors['fox']
similarities = {word: cosine_similarity(fox_vec, vec) for word, vec in word_vectors.items()}
for word, score in sorted(similarities.items(), key=lambda x: -x[1])[:5]:
    print(f"{word}: {score:.3f}")

Starting training...
Epoch 1/200 	 Loss: 4.1957
Epoch 2/200 	 Loss: 3.5344
Epoch 3/200 	 Loss: 3.0504
Epoch 4/200 	 Loss: 2.5162
Epoch 5/200 	 Loss: 2.2120
Epoch 6/200 	 Loss: 2.1002
Epoch 7/200 	 Loss: 2.0428
Epoch 8/200 	 Loss: 1.8061
Epoch 9/200 	 Loss: 1.6945
Epoch 10/200 	 Loss: 1.7851
Epoch 11/200 	 Loss: 1.5748
Epoch 12/200 	 Loss: 1.3482
Epoch 13/200 	 Loss: 1.4505
Epoch 14/200 	 Loss: 1.7298
Epoch 15/200 	 Loss: 0.9473
Epoch 16/200 	 Loss: 1.5695
Epoch 17/200 	 Loss: 1.4923
Epoch 18/200 	 Loss: 1.0521
Epoch 19/200 	 Loss: 1.1886
Epoch 20/200 	 Loss: 1.4126
Epoch 21/200 	 Loss: 0.9736
Epoch 22/200 	 Loss: 1.2919
Epoch 23/200 	 Loss: 1.2413
Epoch 24/200 	 Loss: 1.3046
Epoch 25/200 	 Loss: 1.2233
Epoch 26/200 	 Loss: 1.6594
Epoch 27/200 	 Loss: 1.2876
Epoch 28/200 	 Loss: 1.2977
Epoch 29/200 	 Loss: 1.3733
Epoch 30/200 	 Loss: 0.9327
Epoch 31/200 	 Loss: 0.5775
Epoch 32/200 	 Loss: 1.3267
Epoch 33/200 	 Loss: 0.9460
Epoch 34/200 	 Loss: 1.0236
Epoch 35/200 	 Loss: 1.4108
Epoch 36